In [0]:
spark

In [0]:
df = spark.read.csv(
    "/Volumes/students/default/filestore/retail_orders.csv",
    header=True,
    inferSchema=True
)

df.printSchema()
print("Number of rows", df.count())

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price_krw: integer (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- sales_amount_krw: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- returned: string (nullable = true)
 |-- rating: integer (nullable = true)

Number of rows 120


In [0]:
from pyspark.sql import functions as F

orders = (
    df
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price_krw", F.col("unit_price_krw").cast("double"))
    .withColumn("discount_rate", F.col("discount_rate").cast("double"))
    .withColumn("sales_amount_krw", F.col("sales_amount_krw").cast("double"))
    .withColumn("rating", F.col("rating").cast("int"))
)

orders.printSchema()
display(orders.limit(10))

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price_krw: double (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- sales_amount_krw: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- returned: string (nullable = true)
 |-- rating: integer (nullable = true)



order_id,order_date,customer_id,city,customer_segment,product_id,product_name,category,quantity,unit_price_krw,discount_rate,sales_amount_krw,channel,payment_method,returned,rating
T0001,2026-01-04,C011,Yongin,Basic,P002,Smartphone,Electronics,3,950000.0,0.0,2850000.0,Mobile App,Credit Card,No,null
T0002,2026-03-11,C011,Yongin,Basic,P012,Monitor,Electronics,1,390000.0,0.05,370500.0,In-Store,KakaoPay,Yes,3
T0003,2026-03-19,C004,Daegu,Standard,P009,Pen Set,Stationery,1,12000.0,0.0,12000.0,Online,Naver Pay,No,null
T0004,2026-03-17,C004,Daegu,Standard,P008,Notebook,Stationery,1,6000.0,0.0,6000.0,Mobile App,Naver Pay,No,4
T0005,2026-01-20,C006,Gwangju,Standard,P005,Desk,Furniture,1,310000.0,0.0,310000.0,In-Store,Credit Card,No,5
T0006,2026-03-19,C006,Gwangju,Standard,P006,Coffee Beans,Grocery,1,25000.0,0.0,25000.0,Online,Credit Card,No,5
T0007,2026-03-12,C007,Suwon,Basic,P002,Smartphone,Electronics,1,950000.0,0.15,807500.0,Mobile App,KakaoPay,No,4
T0008,2026-01-06,C012,Anyang,Standard,P002,Smartphone,Electronics,2,950000.0,0.0,1900000.0,In-Store,Credit Card,No,4
T0009,2026-02-28,C007,Suwon,Basic,P005,Desk,Furniture,2,310000.0,0.0,620000.0,Online,KakaoPay,No,4
T0010,2026-03-31,C011,Yongin,Basic,P005,Desk,Furniture,4,310000.0,0.15,1054000.0,Mobile App,KakaoPay,Yes,null


In [0]:
df.createOrReplaceTempView("orders")

In [0]:
%sql
SELECT *
FROM orders
LIMIT 10;


order_id,order_date,customer_id,city,customer_segment,product_id,product_name,category,quantity,unit_price_krw,discount_rate,sales_amount_krw,channel,payment_method,returned,rating
T0001,2026-01-04,C011,Yongin,Basic,P002,Smartphone,Electronics,3,950000,0.0,2850000.0,Mobile App,Credit Card,No,null
T0002,2026-03-11,C011,Yongin,Basic,P012,Monitor,Electronics,1,390000,0.05,370500.0,In-Store,KakaoPay,Yes,3
T0003,2026-03-19,C004,Daegu,Standard,P009,Pen Set,Stationery,1,12000,0.0,12000.0,Online,Naver Pay,No,null
T0004,2026-03-17,C004,Daegu,Standard,P008,Notebook,Stationery,1,6000,0.0,6000.0,Mobile App,Naver Pay,No,4
T0005,2026-01-20,C006,Gwangju,Standard,P005,Desk,Furniture,1,310000,0.0,310000.0,In-Store,Credit Card,No,5
T0006,2026-03-19,C006,Gwangju,Standard,P006,Coffee Beans,Grocery,1,25000,0.0,25000.0,Online,Credit Card,No,5
T0007,2026-03-12,C007,Suwon,Basic,P002,Smartphone,Electronics,1,950000,0.15,807500.0,Mobile App,KakaoPay,No,4
T0008,2026-01-06,C012,Anyang,Standard,P002,Smartphone,Electronics,2,950000,0.0,1900000.0,In-Store,Credit Card,No,4
T0009,2026-02-28,C007,Suwon,Basic,P005,Desk,Furniture,2,310000,0.0,620000.0,Online,KakaoPay,No,4
T0010,2026-03-31,C011,Yongin,Basic,P005,Desk,Furniture,4,310000,0.15,1054000.0,Mobile App,KakaoPay,Yes,null


In [0]:
%sql
SELECT order_id, order_date, city, product_name, category, quantity, sales_amount_krw
FROM orders
WHERE category = 'Electronics' AND returned = 'No'
ORDER BY sales_amount_krw DESC
LIMIT 10;

order_id,order_date,city,product_name,category,quantity,sales_amount_krw
T0022,2026-01-15,Ulsan,Laptop,Electronics,4,4800000.0
T0094,2026-03-08,Jeju,Laptop,Electronics,4,4800000.0
T0114,2026-01-30,Seoul,Smartphone,Electronics,5,4275000.0
T0038,2026-03-28,Sejong,Smartphone,Electronics,4,3610000.0
T0060,2026-03-27,Suwon,Smartphone,Electronics,4,3420000.0
T0001,2026-01-04,Yongin,Smartphone,Electronics,3,2850000.0
T0051,2026-02-15,Yongin,Smartphone,Electronics,3,2707500.0
T0108,2026-02-22,Incheon,Laptop,Electronics,2,2160000.0
T0101,2026-01-07,Daejeon,Laptop,Electronics,2,2040000.0
T0115,2026-01-23,Sejong,Laptop,Electronics,2,2040000.0


In [0]:
%sql
SELECT
    category,
    COUNT(*) AS number_of_orders,
    SUM(quantity) AS total_units,
    ROUND(SUM(sales_amount_krw), 0) AS total_sales_krw,
    ROUND(AVG(sales_amount_krw), 0) AS avg_order_value_krw
FROM orders
GROUP BY category
ORDER BY total_sales_krw DESC;

category,number_of_orders,total_units,total_sales_krw,avg_order_value_krw
Electronics,43,94,5.75235E7,1337756.0
Furniture,29,50,1.2156E7,419172.0
Accessories,11,25,1425600.0,129600.0
Grocery,17,42,838950.0,49350.0
Stationery,20,37,312300.0,15615.0


In [0]:
%sql
SELECT
    customer_segment,
    channel,
    COUNT(*) AS orders,
    ROUND(SUM(sales_amount_krw), 0) AS revenue_krw
FROM orders
GROUP BY customer_segment, channel
ORDER BY customer_segment, revenue_krw DESC;

customer_segment,channel,orders,revenue_krw
Basic,Online,11,9546200.0
Basic,Mobile App,12,8100600.0
Basic,In-Store,13,5839100.0
Premium,Mobile App,11,1.11809E7
Premium,Online,10,8986500.0
Premium,In-Store,12,7344200.0
Standard,Mobile App,17,1.06331E7
Standard,In-Store,15,5603500.0
Standard,Online,19,5022250.0


In [0]:
%sql
SELECT
    DATE_TRUNC('month', order_date) AS order_month,
    COUNT(*) AS orders,
    ROUND(SUM(sales_amount_krw), 0) AS monthly_revenue_krw
FROM orders
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY order_month;

order_month,orders,monthly_revenue_krw
2026-01-01T00:00:00.000Z,38,2.48986E7
2026-02-01T00:00:00.000Z,39,1.88794E7
2026-03-01T00:00:00.000Z,43,2.847835E7


In [0]:
%sql
SELECT
    order_id,
    product_name,
    sales_amount_krw,
    CASE
        WHEN sales_amount_krw >= 1000000 THEN 'High value'
        WHEN sales_amount_krw >= 200000 THEN 'Medium value'
        ELSE 'Low value'
    END AS order_value_group
FROM orders
ORDER BY sales_amount_krw DESC
LIMIT 15;

order_id,product_name,sales_amount_krw,order_value_group
T0022,Laptop,4800000.0,High value
T0094,Laptop,4800000.0,High value
T0114,Smartphone,4275000.0,High value
T0038,Smartphone,3610000.0,High value
T0060,Smartphone,3420000.0,High value
T0001,Smartphone,2850000.0,High value
T0051,Smartphone,2707500.0,High value
T0108,Laptop,2160000.0,High value
T0115,Laptop,2040000.0,High value
T0101,Laptop,2040000.0,High value


In [0]:
%sql
SELECT
    count(*) as total_rows,
    sum(case when rating is null then 1 else 0 end) as missing_ratings
FROM orders;

total_rows,missing_ratings
120,22


In [0]:
%sql
SELECT
    product_name,
    rating,
    coalesce(rating, 0) as rating_filled_with_zero
from orders
where rating is NULL
limit 10;

product_name,rating,rating_filled_with_zero
Smartphone,null,0
Pen Set,null,0
Desk,null,0
Monitor,null,0
Monitor,null,0
Desk,null,0
Headphones,null,0
Office Chair,null,0
Office Chair,null,0
Headphones,null,0


In [0]:
customers_data = [
    ("C001", "Seoul", "Premium"), ("C002", "Busan", "Standard"), ("C003", "Incheon", "Premium"), ("C004", "Daegu", "Standard"),
    ("C005", "Daejeon", "Basic"), ("C006", "Gwangju", "Standard"), ("C007", "Suwon", "Basic"), ("C008", "Ulsan", "Premium"),
    ("C009", "Jeju", "Standard"), ("C010", "Sejong", "Premium"), ("C011", "Yongin", "Basic"), ("C012", "Anyang", "Standard")
]

products_data = [
    ("P001", "Laptop", "Electronics"), ("P002", "Smartphone", "Electronics"), ("P003", "Headphones", "Electronics"),
    ("P004", "Office Chair", "Furniture"), ("P005", "Desk", "Furniture"), ("P006", "Coffee Beans", "Grocery"),
    ("P007", "Green Tea", "Grocery"), ("P008", "Notebook", "Stationery"), ("P009", "Pen Set", "Stationery"),
    ("P010", "Backpack", "Accessories"), ("P011", "Water Bottle", "Accessories"), ("P012", "Monitor", "Electronics")
]

customers = spark.createDataFrame(customers_data, ["customer_id", "home_city", "segment_from_dim"])
products = spark.createDataFrame(products_data, ["product_id", "product_name_dim", "category_from_dim"])

customers.createOrReplaceTempView("customers")
products.createOrReplaceTempView("products")

In [0]:
%sql
SELECT
    o.order_id,
    o.order_date,
    c.home_city,
    c.segment_from_dim,
    p.product_name_dim,
    p.category_from_dim,
    o.quantity,
    o.sales_amount_krw
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
ORDER BY o.order_date
LIMIT 20;

order_id,order_date,home_city,segment_from_dim,product_name_dim,category_from_dim,quantity,sales_amount_krw
T0034,2026-01-01,Incheon,Premium,Green Tea,Grocery,5,90000.0
T0119,2026-01-01,Suwon,Basic,Pen Set,Stationery,2,21600.0
T0063,2026-01-01,Suwon,Basic,Pen Set,Stationery,1,12000.0
T0081,2026-01-03,Busan,Standard,Notebook,Stationery,3,18000.0
T0001,2026-01-04,Yongin,Basic,Smartphone,Electronics,3,2850000.0
T0057,2026-01-05,Sejong,Premium,Desk,Furniture,1,294500.0
T0008,2026-01-06,Anyang,Standard,Smartphone,Electronics,2,1900000.0
T0039,2026-01-06,Sejong,Premium,Backpack,Accessories,2,161500.0
T0030,2026-01-07,Suwon,Basic,Notebook,Stationery,2,10200.0
T0101,2026-01-07,Daejeon,Basic,Laptop,Electronics,2,2040000.0


In [0]:
%sql
WITH category_sales AS (
    SELECT
        category,
        SUM(sales_amount_krw) AS revenue_krw
    FROM orders
    WHERE returned = 'No'
    GROUP BY category
),
total_sales AS (
    SELECT SUM(revenue_krw) AS total_revenue_krw
    FROM category_sales
)
SELECT
    category,
    ROUND(revenue_krw, 0) AS revenue_krw,
    ROUND(100 * revenue_krw / total_revenue_krw, 2) AS revenue_share_percent
FROM category_sales
CROSS JOIN total_sales
ORDER BY revenue_krw DESC;

category,revenue_krw,revenue_share_percent
Electronics,5.55915E7,81.53
Furniture,1.0172E7,14.92
Accessories,1340600.0,1.97
Grocery,773650.0,1.13
Stationery,306600.0,0.45


In [0]:
%sql
SELECT
    order_id,
    category,
    product_name,
    sales_amount_krw,
    RANK() OVER (PARTITION BY category ORDER BY sales_amount_krw DESC) AS rank_within_category
FROM orders
QUALIFY rank_within_category <= 3
ORDER BY category, rank_within_category;

order_id,category,product_name,sales_amount_krw,rank_within_category
T0015,Accessories,Backpack,323000.0,1
T0110,Accessories,Backpack,255000.0,2
T0087,Accessories,Backpack,229500.0,3
T0022,Electronics,Laptop,4800000.0,1
T0094,Electronics,Laptop,4800000.0,1
T0114,Electronics,Smartphone,4275000.0,3
T0025,Furniture,Desk,1317500.0,1
T0010,Furniture,Desk,1054000.0,2
T0066,Furniture,Desk,930000.0,3
T0055,Grocery,Coffee Beans,100000.0,1


In [0]:
# the total revenue for each product category from non-returned orders and sort by revenue
top_categories_df = spark.sql("""
    SELECT category, SUM(sales_amount_krw) AS revenue_krw
    FROM orders
    WHERE returned = 'No'
    GROUP BY category
    ORDER BY revenue_krw DESC
""")

display(top_categories_df)

category,revenue_krw
Electronics,5.55915E7
Furniture,1.0172E7
Accessories,1340600.0
Grocery,773650.0
Stationery,306600.0


In [0]:
# Save the actual DataFrame you created in the previous step
top_categories_df.write.mode('overwrite').format('delta').saveAsTable('category_revenue_summary')

In [0]:
%sql
SELECT * FROM category_revenue_summary
ORDER BY revenue_krw DESC;

category,revenue_krw
Electronics,5.55915E7
Furniture,1.0172E7
Accessories,1340600.0
Grocery,773650.0
Stationery,306600.0


In [0]:
%sql
-- Find the top five cities by total revenue.
SELECT city, ROUND(SUM(sales_amount_krw), 0) AS revenue_krw
FROM orders
GROUP BY city
ORDER BY revenue_krw DESC
LIMIT 5;

city,revenue_krw
Yongin,1.2658E7
Ulsan,9298100.0
Jeju,7386550.0
Sejong,7218300.0
Busan,6464100.0


In [0]:
%sql
-- Compute the return rate for each product category.
SELECT
    category,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN returned = 'Yes' THEN 1 ELSE 0 END) AS returned_orders,
    ROUND(100 * SUM(CASE WHEN returned = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS return_rate_percent
FROM orders
GROUP BY category
ORDER BY return_rate_percent DESC;

category,total_orders,returned_orders,return_rate_percent
Grocery,17,2,11.76
Accessories,11,1,9.09
Electronics,43,3,6.98
Furniture,29,2,6.9
Stationery,20,1,5.0
